In [ ]:
import math
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import Dataset,Subset,DataLoader,random_split
from tqdm.notebook import tqdm
from torch.optim.lr_scheduler import ReduceLROnPlateau
import itertools

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')

# Architecture

In [ ]:
class Discriminator(nn.Module):
  def __init__(self,in_features):
    super().__init__()
    self.disc = nn.Sequential(
        nn.Linear(in_features,128),
        nn.LeakyReLU(0.1),
        nn.Linear(128,1),
        nn.Sigmoid()
    )

  def forward(self,x):
    return self.disc(x)

In [ ]:
class Generator(nn.Module):
  def __init__(self,z_dim,img_dim):
    super().__init__()
    self.gen = nn.Sequential(
        nn.Linear(z_dim,256),
        nn.LeakyReLU(0.1),
        nn.Linear(256,img_dim),
        nn.Tanh()
    )

  def forward(self,x):
    return self.gen(x)

In [ ]:
'''
class GAN(nn.Module):
  def __init__(self,cfg):
    super().__init__()
    self.cfg=cfg
    self.disc = Discriminator(cfg.img_dim)
    self.gen = Generator(cfg.z_dim,cfg.img_dim)

  def generate(self):
    z=torch.randn(self.cfg.batch_size,self.cfg.z_dim).to(device)
    return self.gen(z)

  def forward(self,x):
    return self.disc(x)*(1-self.disc(self.generate()))
    '''

  # 이렇게 하는것 보단 따로따로 하는게 더 편할 듯


# HyperParameter and Config

In [ ]:
class GANConfig:
  z_dim: int = 64
  img_dim: int = 28*28*1

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
lr=3e-4
batch_size=32
epochs = 50

# Dataset

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,),(0.3081,))
])

In [ ]:
dataset=datasets.MNIST(
    root='dataset/',
    train=True,
    transform=transform,
    download=True
)

In [ ]:
loader=DataLoader(dataset,batch_size=batch_size,shuffle=True)

# Model

In [ ]:
disc=Discriminator(GANConfig.img_dim).to(device)
gen=Generator(GANConfig.z_dim,GANConfig.img_dim).to(device)

In [ ]:
fixed_noise=torch.randn((batch_size,z_dim)).to(device)

In [ ]:
opt_disc=optim.Adam(disc.parameters(),lr=lr)
opt_gen=optim.Adam(gen.parameters(),lr=lr)
criterion=nn.BCELoss()

# Training

In [ ]:
for epoch in range(epochs):
  for batch_idx, (real,_) in enumerate(loader):
    real=real.view(-1,28*28).to(device)
    batch_size=real.shape[0]

    noise=torch.randn(batch_size,GANConfig.z_dim).to(device)
    fake=gen(noise)

    disc_real=disc(real).view(-1)
    disc_fake=disc(fake.detach).view(-1)

    lossD_real=criterion(disc_real,torch.ones_like(disc_real))
    lossD_fake=criterion(disc_fake,torch.zeros_like(disc_real))
    lossD=(lossD_real+lossD_fake)/2
    disc.zero_grad()
    lossD.backward(retrain_graph=True)
    opt_disc.step()

    output=disc(fake).view(-1)
    lossG=criterion(output,torch.ones_like(output))
    gen.zero_grad()
    lossG.backward()
    opt_gen.step()

    # 뒤에 생성과정 그리기